# Data Preparation — Churn Prediction

Pipeline completo: copia de tablas → feature engineering → guardado.

Decisiones de features documentadas en `feature_creation.md`.

In [ ]:
%run "./env_setup.py"
import pandas as pd
import os

## 1. Schema y copia de tablas

In [ ]:
SCHEMA = username

agent.execute_ddl(f"DROP SCHEMA IF EXISTS {SCHEMA} CASCADE")
agent.execute_ddl(f"CREATE SCHEMA {SCHEMA}")
print(f"Schema '{SCHEMA}' recreado desde cero.")

for tabla in ['customers', 'products', 'stores', 'sales', 'new_sales']:
    agent.execute_ddl(f"""
        CREATE TABLE {SCHEMA}.{tabla} AS SELECT * FROM public.{tabla}
    """)
    print(f"  ✓ {SCHEMA}.{tabla}")

## 2. Carga de datos

In [ ]:
df_train = agent.execute_dml(f"""
    SELECT
        s.code, s.sales_date, s.base_date, s.customer_id,
        s.pvp, s.forma_pago, s.coste_venta_no_impuestos, s.margen_eur,
        s.extension_garantia, s.seguro_bateria_largo_plazo,
        s.mantenimiento_gratuito, s.en_garantia, s.revisiones,
        s.km_ultima_revision, s.km_medio_por_revision,
        s.encuesta_cliente_zona_taller, s.queja,
        s.fin_garantia, s.days_last_service, s.lead_compra, s.fue_lead,
        s.churn_400,
        c.edad, c.genero, c.renta_media_estimada, c.status_social,
        p.modelo, p.fuel, p.equipamiento, p.kw,
        st.zona
    FROM {SCHEMA}.sales s
    JOIN {SCHEMA}.customers c ON s.customer_id = c.customer_id
    JOIN {SCHEMA}.products p  ON s.id_producto  = p.id_producto
    JOIN {SCHEMA}.stores st   ON s.tienda_desc   = st.tienda_desc
""")

df_scoring = agent.execute_dml(f"""
    SELECT
        ns.code, ns.sales_date, ns.base_date, ns.customer_id,
        ns.pvp, ns.forma_pago, ns.coste_venta_no_impuestos, ns.margen_eur,
        ns.extension_garantia, ns.seguro_bateria_largo_plazo,
        ns.mantenimiento_gratuito, ns.en_garantia, ns.revisiones,
        ns.km_ultima_revision, ns.km_medio_por_revision,
        ns.encuesta_cliente_zona_taller, ns.queja,
        ns.fin_garantia, ns.lead_compra,
        c.edad, c.genero, c.renta_media_estimada, c.status_social,
        p.modelo, p.fuel, p.equipamiento, p.kw,
        st.zona
    FROM {SCHEMA}.new_sales ns
    JOIN {SCHEMA}.customers c ON ns.customer_id = c.customer_id
    JOIN {SCHEMA}.products p  ON ns.id_producto  = p.id_producto
    JOIN {SCHEMA}.stores st   ON ns.tienda_desc   = st.tienda_desc
""")

for df in [df_train, df_scoring]:
    df['sales_date']  = pd.to_datetime(df['sales_date'])
    df['base_date']   = pd.to_datetime(df['base_date'])
    df['fin_garantia'] = pd.to_datetime(df['fin_garantia'])

# Columnas solo en sales (no en new_sales): rellenar con NaN para scoring
for col in ['days_last_service', 'fue_lead']:
    if col not in df_scoring.columns:
        df_scoring[col] = pd.NA

print(f"Train:   {df_train.shape[0]:,} filas × {df_train.shape[1]} cols")
print(f"Scoring: {df_scoring.shape[0]:,} filas × {df_scoring.shape[1]} cols")

In [ ]:
## 2b. Agregados históricos por cliente (RFM + intervalos)
# Se computan desde el histórico completo de sales y se mergean a ambos dfs.
# frequency_total, tenure_days, ticket_avg, service_interval_mean/std

df_cust_agg = agent.execute_dml(f"""
    WITH sales_lag AS (
        SELECT
            customer_id,
            sales_date,
            pvp,
            LAG(sales_date) OVER (PARTITION BY customer_id ORDER BY sales_date) AS prev_date
        FROM {SCHEMA}.sales
    )
    SELECT
        customer_id,
        COUNT(*)                                               AS frequency_total,
        MIN(sales_date)                                        AS first_sale_date,
        AVG(pvp)                                               AS ticket_avg,
        AVG(
            CASE WHEN prev_date IS NOT NULL
                 THEN (sales_date - prev_date)::float END
        )                                                      AS service_interval_mean_days,
        STDDEV_SAMP(
            CASE WHEN prev_date IS NOT NULL
                 THEN (sales_date - prev_date)::float END
        )                                                      AS service_interval_std_days
    FROM sales_lag
    GROUP BY customer_id
""")
df_cust_agg['first_sale_date'] = pd.to_datetime(df_cust_agg['first_sale_date'])

df_train   = df_train.merge(df_cust_agg, on='customer_id', how='left')
df_scoring = df_scoring.merge(df_cust_agg, on='customer_id', how='left')

print(f"Train:   {df_train.shape[0]:,} × {df_train.shape[1]} cols")
print(f"Scoring: {df_scoring.shape[0]:,} × {df_scoring.shape[1]} cols")

## 3. Feature Engineering

22 features seleccionadas tras el análisis tabla por tabla (ver `feature_creation.md`).

Las variables categóricas se mantienen como strings — el **encoding se realiza en el pipeline de modelado**, no aquí.

In [ ]:
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── Temporal ──────────────────────────────────────────────────────────────
    base_date = df['sales_date'].max()
    df['dias_desde_compra']      = (base_date - df['sales_date']).dt.days
    df['tenure_days']            = (df['sales_date'] - df['first_sale_date']).dt.days
    df['garantia_dias_restantes'] = (df['fin_garantia'] - base_date).dt.days

    # ── Económicas ────────────────────────────────────────────────────────────
    df['margen_relativo']     = df['margen_eur'] / df['pvp'].replace(0, pd.NA)
    df['margen_eur_negativo'] = (df['margen_eur'] < 0).astype(int)

    # ── Producto ──────────────────────────────────────────────────────────────
    df['es_electrico'] = (df['fuel'] != 'HÍBRIDO').astype(int)

    # ── Garantía: dos flags (el salto NO→SI es +0.5pp; SI→SI-Financiera es -4.4pp)
    df['ext_garantia_tiene']      = (df['extension_garantia'] != 'NO').astype(int)
    df['ext_garantia_financiera'] = df['extension_garantia'].str.contains('Financiera', na=False).astype(int)

    # ── Servicios — BUG FIX: mantenimiento_gratuito es 0/4, no 'SI'/'NO'
    df['mantenimiento_gratuito']     = (df['mantenimiento_gratuito'].astype(str) != '0').astype(int)
    df['seguro_bateria_largo_plazo'] = (df['seguro_bateria_largo_plazo'] == 'SI').astype(int)
    df['en_garantia']                = (df['en_garantia'] == 'SI').astype(int)
    df['sin_revisiones']             = (df['revisiones'] == 0).astype(int)

    # ── Renta ─────────────────────────────────────────────────────────────────
    df['renta_desconocida'] = (df['renta_media_estimada'] == 0).astype(int)

    # ── Satisfacción / quejas ─────────────────────────────────────────────────
    df['tiene_queja'] = (df['queja'] == 'SI').astype(int)

    return df


df_train   = feature_engineering(df_train)
df_scoring = feature_engineering(df_scoring)
print("Features creadas.")

In [ ]:
def tratar_nulos(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Demográficas
    df['genero']               = df['genero'].fillna('Desconocido')
    df['status_social']        = df['status_social'].fillna('Sin_dato')
    df['renta_media_estimada'] = df['renta_media_estimada'].fillna(0)
    df['margen_relativo']      = df['margen_relativo'].fillna(0)

    # Satisfacción: NaN → media neutral (no sabemos si hubo encuesta o no)
    df['encuesta_cliente_zona_taller'] = df['encuesta_cliente_zona_taller'].fillna(
        df['encuesta_cliente_zona_taller'].median()
    )

    # Odómetro: NaN → 0 (sin revisiones, sin dato)
    df['km_ultima_revision']    = df['km_ultima_revision'].fillna(0)
    df['km_medio_por_revision'] = df['km_medio_por_revision'].fillna(0)

    # Garantía restante: NaN si fin_garantia era NULL → -999 (sin garantía registrada)
    df['garantia_dias_restantes'] = df['garantia_dias_restantes'].fillna(-999)

    # Columnas solo en sales, NaN en scoring → 0 (no registrado / cliente nuevo)
    df['days_last_service'] = df['days_last_service'].fillna(0)
    df['fue_lead']          = df['fue_lead'].fillna(0)
    df['lead_compra']       = df['lead_compra'].fillna(0)

    # Agregados por cliente: NaN si cliente sin histórico → 0
    df['frequency_total']            = df['frequency_total'].fillna(0)
    df['tenure_days']                = df['tenure_days'].fillna(0)
    df['ticket_avg']                 = df['ticket_avg'].fillna(df['pvp'])
    df['service_interval_mean_days'] = df['service_interval_mean_days'].fillna(0)
    df['service_interval_std_days']  = df['service_interval_std_days'].fillna(0)

    return df


df_train   = tratar_nulos(df_train)
df_scoring = tratar_nulos(df_scoring)

restantes = df_train.isnull().sum()
restantes = restantes[restantes > 0]
print("NaN en train:", restantes.to_dict() if len(restantes) else "Ninguno ✓")

In [ ]:
FEATURES = [
    # ── Económicas ────────────────────────────────────────────────────────────
    'pvp', 'margen_relativo', 'margen_eur_negativo', 'coste_venta_no_impuestos',
    'forma_pago',

    # ── Temporales ────────────────────────────────────────────────────────────
    'dias_desde_compra',
    'tenure_days',                   # días desde 1ª compra del cliente
    'days_last_service',             # días desde último servicio (solo en sales)

    # ── Garantía ──────────────────────────────────────────────────────────────
    'en_garantia', 'ext_garantia_tiene', 'ext_garantia_financiera',
    'garantia_dias_restantes',       # días restantes de garantía en base_date

    # ── Servicios / mantenimiento ──────────────────────────────────────────────
    'mantenimiento_gratuito', 'seguro_bateria_largo_plazo',
    'sin_revisiones', 'revisiones',

    # ── Odómetro / uso ────────────────────────────────────────────────────────
    'km_ultima_revision',            # proxy odómetro último registro
    'km_medio_por_revision',         # km recorridos entre revisiones

    # ── Satisfacción / quejas ─────────────────────────────────────────────────
    'encuesta_cliente_zona_taller',  # puntuación encuesta taller (SERVQUAL proxy)
    'tiene_queja',                   # reclamación registrada (0/1)

    # ── Comportamiento comercial ───────────────────────────────────────────────
    'lead_compra',                   # días desde lead a compra
    'fue_lead',                      # vino por campaña de leads (solo en sales)

    # ── RFM cross-cliente ─────────────────────────────────────────────────────
    'frequency_total',               # nº compras históricas del cliente
    'ticket_avg',                    # ticket medio histórico del cliente
    'service_interval_mean_days',    # intervalo medio entre compras
    'service_interval_std_days',     # irregularidad del intervalo

    # ── Cliente ───────────────────────────────────────────────────────────────
    'edad', 'genero', 'renta_media_estimada', 'renta_desconocida', 'status_social',

    # ── Producto ──────────────────────────────────────────────────────────────
    'modelo', 'equipamiento', 'es_electrico', 'kw',

    # ── Geografía ─────────────────────────────────────────────────────────────
    'zona',                          # zona del concesionario
]

df_train_fe   = df_train[FEATURES].copy()
df_train_fe['churn'] = (df_train['churn_400'] == 'Y').astype(int)

df_scoring_fe = df_scoring[['code'] + FEATURES].copy()

print(f"Train:   {df_train_fe.shape[0]:,} filas × {df_train_fe.shape[1]} cols | churn rate: {df_train_fe['churn'].mean():.1%}")
print(f"Scoring: {df_scoring_fe.shape[0]:,} filas × {df_scoring_fe.shape[1]} cols")
print(f"\nFeatures ({len(FEATURES)}):\n{FEATURES}")

In [ ]:
print(df_train_fe.dtypes.to_string())
print(f"\nNaN: {df_train_fe.isnull().sum().sum()}")

## 4. Guardado

- `barrechee.features_train` / `train.csv` — 58.049 filas con `churn`. Split train/test en modeling.
- `barrechee.features_scoring` / `scoring.csv` — 10.000 clientes nuevos sin etiqueta.

In [ ]:
import os

agent.execute_ddl(f"DROP TABLE IF EXISTS {SCHEMA}.features_train")
agent.execute_ddl(f"DROP TABLE IF EXISTS {SCHEMA}.features_scoring")
agent.append_to_postgres(df_train_fe,   'features_train')
agent.append_to_postgres(df_scoring_fe, 'features_scoring')

os.makedirs("data/warehouse", exist_ok=True)
df_train_fe.to_csv("data/warehouse/train.csv",     index=False)
df_scoring_fe.to_csv("data/warehouse/scoring.csv", index=False)

print(f"✓ features_train   / train.csv   — {df_train_fe.shape[0]:,} × {df_train_fe.shape[1]}")
print(f"✓ features_scoring / scoring.csv — {df_scoring_fe.shape[0]:,} × {df_scoring_fe.shape[1]}")